# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading, exploring, and processing the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** In Croissant, all entities are referenced by their `@id` (identifier), which uniquely identifies record sets, fields, and columns.

In [ ]:
# List all record sets in the dataset and their @id
record_sets = list(dataset.record_sets)

print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    # List fields in each record set by @id
    if 'field' in rs:
        if isinstance(rs['field'], list):
            for f in rs['field']:
                if isinstance(f, dict):
                    print(f"  Field: {f.get('@id', str(f))}")
                else:
                    print(f"  Field: {f}")
        elif isinstance(rs['field'], dict):
            print(f"  Field: {rs['field'].get('@id', str(rs['field']))}")
        else:
            print(f"  Field: {rs['field']}")
    else:
        print("  No fields found.")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s identified above.

> **Tip:** Replace the sample record set and field `@id`s with those found in your dataset when running the notebook.

In [ ]:
# Example: Extract data from all available record sets into DataFrames
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"\nLoaded {len(df)} records for record set: {record_set_id}")
            print("Fields:", df.columns.tolist())
        else:
            print(f"\nNo records found in record set: {record_set_id}")
    except Exception as e:
        print(f"\nCould not load record set {record_set_id}: {e}")

# Display first few rows of the first DataFrame
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply processing steps, such as filtering records by criteria, normalizing numeric fields, or grouping data by attributes.

> Make sure to update field `@id`s below with real `@id`s from the DataFrames above for actual EDA.

In [ ]:
# Identify a record set and field for numeric analysis
if dataframes:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print(f"\nAvailable fields in {rs_id}:\n", df.columns.tolist())
    
    # Try to automatically pick a numeric field, or replace this with the desired field @id
    candidate_numeric_cols = df.select_dtypes(include=['float', 'int']).columns
    if len(candidate_numeric_cols) > 0:
        numeric_field = candidate_numeric_cols[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records from {rs_id} with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() > 0 else 1)
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Perform groupby on a categorical field, if available
        candidate_group_cols = df.select_dtypes(include=['object']).columns
        if len(candidate_group_cols) > 0:
            group_field = candidate_group_cols[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped filtered data by {group_field}:")
            display(grouped_df.head())
        else:
            print("\nNo categorical fields available for grouping in this record set.")
    else:
        print("No numeric fields found for EDA in this record set.")
else:
    print("No dataframes available for EDA. Please check if dataset contains data.")

## 5. Visualization
Visualize numeric and categorical relationships in the dataset. Below is a generic visualization example for the first available DataFrame and numeric/categorical columns.

*You can extend this with more domain-specific charts as needed.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the numeric field
if dataframes:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    candidate_numeric_cols = df.select_dtypes(include=['float', 'int']).columns
    if len(candidate_numeric_cols) > 0:
        numeric_field = candidate_numeric_cols[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()

        # If a categorical column exists, plot grouped mean
        candidate_group_cols = df.select_dtypes(include=['object']).columns
        if len(candidate_group_cols) > 0:
            group_field = candidate_group_cols[0]
            grouped = df.groupby(group_field)[numeric_field].mean().reset_index()
            plt.figure(figsize=(10,4))
            sns.barplot(x=group_field, y=numeric_field, data=grouped)
            plt.title(f"Mean {numeric_field} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(f"Mean {numeric_field}")
            plt.xticks(rotation=45)
            plt.show()
    else:
        print("No numeric fields available for visualization.")
else:
    print("No dataframes found for visualization.")

## 6. Conclusion

In this notebook, we:
- Loaded metadata and records from a Croissant FAIR<sup>2</sup> dataset using `mlcroissant`.
- Listed available record sets and their fields using `@id` references.
- Extracted data as DataFrames, applied basic EDA (filtering, normalization, grouping).
- Visualized numeric field distributions and relationships to categorical variables.

This workflow serves as a foundation for deeper statistical analysis and hypothesis testing using FAIR datasets in the Croissant format. For advanced modeling or data wrangling, you may leverage additional functionality within the `mlcroissant` and `pandas` ecosystems.